In [3]:
import os, sys
sys.path.append('../')
sys.path.append("/scratch/halmazan/NEXT/testing/notebooks/FOM_creator/")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from pathlib import Path


import importlib
#importlib.reload(FOM)
from tqdm.notebook import tqdm
#importlib.reload(fitf)
from invisible_cities.io.dst_io           import load_dst, load_dsts, df_writer

import functions_HE as func
import FOM_functions_050526      as FOM
import cutting_functions  as cutf
import fitting_functions  as fitf
import plotting_functions as plotf

In [ ]:
CITY = 'THEKLA'

RUN_NUMBER = '15589, 15590, 15591, 15592'
RUN_NUMBER = [int(x) for x in RUN_NUMBER.split(',')]
FOM_TS = ['355018']*len(RUN_NUMBER)
#FOM_TS = ['201025']
TIMESTAMP = FOM_TS
# make directory
pre_dir = '/data/halmazan/NEXT/N100_LPR/'


def collect_data(RUN_NUMBER, TS):
    folder_name = f'{pre_dir}/{RUN_NUMBER}/thekla/{TS}/'
    folder_s = Path(f'{folder_name}')

    tdst_full = []
    for LDC in tqdm(range(1,8)):
        files = list(Path(f'{folder_name}/ldc{LDC}/').rglob('*.h5'))
        print(f'LDC {LDC}')
        for file in tqdm(files):
            try:
                tdst_full.append(load_dst(file, 'Tracking', 'Tracks'))
            except KeyboardInterrupt:
                print('kb inter')
                break
            except Exception as e:
                print(e)

    tdst_full = pd.concat(tdst_full)
    # apply cuts to save everyone time
    low_z = 20
    upp_z  = 1195
    r_lim = 300
    low_e = 1.4
    upp_e = 1.8

    cut_hdst_full, efficiencies = func.apply_cuts(tdst_full, 
                                             lower_z = low_z, 
                                             upper_z = upp_z, 
                                             r_lim   = r_lim, 
                                             lower_e = low_e, 
                                             upper_e = upp_e)
    
    return cut_hdst_full

tdst = []
for RN, TS in zip(RUN_NUMBER, FOM_TS):
    tdst.append(collect_data(RN, TS))

tdst = pd.concat(tdst)

tdst.to_hdf(f'full_dataset_{RUN_NUMBER}_{TIMESTAMP}.h5', key='Tracking/Tracks', mode='w')


  0%|          | 0/7 [00:00<?, ?it/s]

LDC 1


  0%|          | 0/5401 [00:00<?, ?it/s]

group ``/`` does not have a child named ``Tracking``
group ``/`` does not have a child named ``Tracking``
group ``/`` does not have a child named ``Tracking``
group ``/`` does not have a child named ``Tracking``
group ``/`` does not have a child named ``Tracking``
group ``/`` does not have a child named ``Tracking``
group ``/`` does not have a child named ``Tracking``
group ``/`` does not have a child named ``Tracking``
group ``/`` does not have a child named ``Tracking``
group ``/`` does not have a child named ``Tracking``
group ``/`` does not have a child named ``Tracking``
group ``/`` does not have a child named ``Tracking``
group ``/`` does not have a child named ``Tracking``
group ``/`` does not have a child named ``Tracking``
group ``/`` does not have a child named ``Tracking``
group ``/`` does not have a child named ``Tracking``
group ``/`` does not have a child named ``Tracking``
group ``/`` does not have a child named ``Tracking``


In [ ]:
display(tdst)
plt.hist(tdst.groupby('event').energy.sum(), bins = 100)
plt.xlabel('Energy (MeV)')
plt.show()

In [ ]:
if True:
    #######################################################################################
    #########################   CUT PARAMETERS   #####################################
    #######################################################################################
    low_z = 20
    upp_z  = 1195
    r_lim = 300
    low_e = 1.4
    upp_e = 1.8

    cut_hdst, efficiencies = func.apply_cuts(tdst, 
                                             lower_z = low_z, 
                                             upper_z = upp_z, 
                                             r_lim   = r_lim, 
                                             lower_e = low_e, 
                                             upper_e = upp_e)

    display(efficiencies)


    # signal vs background blob 2 distro
    plt.hist(cut_hdst.eblob2.values, label = 'bck', range = [0, 1], bins = 100, alpha = 0.7)

    plt.xlabel('blob 2 energy (MeV)')
    plt.legend()
    plt.show()


        # signal vs background blob 2 distro

    plt.hist(cut_hdst.eblob1.values, label = 'sig', range = [0, 1], bins = 100, alpha = 0.7)
    
    plt.xlabel('blob 1 energy (MeV)')
    plt.legend()
    plt.show()

In [ ]:
cut_list = np.linspace(0, 0.6, 31)
fit_info = dict(bins = 80, fit_range = (1.45, 1.65))

mu_config    = dict(
                value    = 1.59,
                floating = False,
                lower    = 1.4,
                upper    = 1.8)

sigma_config = dict(
                value    = 0.1,
                floating = False,
                lower    = 0)

tau_config   = dict(
                value    = 0.03,
                floating = False,
                lower    = 0,
                upper    = 1)


seeds = dict(
            ns = cut_hdst.event.nunique()/2,
            nb = cut_hdst.event.nunique()/2,
            signal = {'mu_config': mu_config,
                      'sigma_config': sigma_config},
            background = {'lambda_config': tau_config})

# mkdir
Path(f'./output/{RUN_NUMBER}_{FOM_TS}').mkdir(parents=True, exist_ok=True)
x = FOM.FOM(cut_hdst, fitf.gaussian_no_N, fitf.exp_no_N, seeds = seeds, fitting_info = fit_info, plot = True, output_path = f'./output/{RUN_NUMBER}_{FOM_TS}')



In [ ]:
# load in the csv and visualise
import csv
import matplotlib.pyplot as plt

def read_FOM_csv(path):
    data = np.genfromtxt(path, delimiter=',')
    n    = len(data) // 5

    return dict(
        cut_list = data[0*n : 1*n, 0],
        fom      = data[0*n : 1*n, 1],
        fom_err  = data[1*n : 2*n, 1],
        ns_l     = data[2*n : 3*n, 0],
        nb_l     = data[2*n : 3*n, 1],
        e        = data[3*n : 4*n, 0],
        e_err    = data[3*n : 4*n, 1],
        b        = data[4*n : 5*n, 0],
        b_err    = data[4*n : 5*n, 1],
    )

FOM_info = read_FOM_csv(f'output/{RUN_NUMBER}_{FOM_TS}/FOM.csv')

FOM_clean = FOM_info['fom']
FOM_clean[(FOM_clean > 5) | (FOM_clean < 0)] = 0

plt.plot(FOM_info['cut_list'], FOM_clean)
plt.xlabel('blob energy (MeV)')
plt.title(f'{RUN_NUMBER}-{TIMESTAMP}')
plt.ylabel('FOM')
plt.grid()
plt.show()
max_fom_index = np.argmax(FOM_clean)
max_fom       = FOM_clean[max_fom_index]
cut_pos       = FOM_info['cut_list'][max_fom_index]
print(f'Max FOM as {max_fom:.2f} at {cut_pos:.2f} MeV')
print(f'Signal events at peak: {FOM_info['ns_l'][max_fom_index]:.2f}')
print(f'Background events at peak: {FOM_info['nb_l'][max_fom_index]:.2f}')
print(f'Initial signal events: {FOM_info['ns_l'][0]:.2f}')
print(f'Initial background events: {FOM_info['nb_l'][0]:.2f}')